# Find the digital gain for a raw example frame

This notebook locates a completely raw target frame within naturally ordered raw world-camera chunks and returns the corresponding digital-gain value from the frame-aligned metadata. It supports both the modern `AGCDgain` column and the legacy `Dgain` column. Raw chunks are examined one at a time to limit memory use. Matching is exact: the target and stored raw frame must have identical shapes and pixel values.

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import cv2
from natsort import natsorted
import os


def locate_project_root(start: Path | None = None) -> Path:
    """Locate the lightLoggerAnalysis repository from a notebook launch path."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "code").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the lightLoggerAnalysis repository.")


PROJECT_ROOT = locate_project_root()
CHUNK_IO_PATH = PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"
WORLD_UTIL_PATH = PROJECT_ROOT / "code" / "library" / "sensor_utility"

for module_path in (CHUNK_IO_PATH, WORLD_UTIL_PATH):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

import chunk_io
import world_util

chunk_io = importlib.reload(chunk_io)
world_util = importlib.reload(world_util)

In [ ]:
def find_DGain(
    raw_chunks_path: str,
    target: np.ndarray,
    verbose: bool = True,
) -> tuple[float, int] | None:
    """Return the digital gain and global index of an exact target-frame match.

    Args:
        raw_chunks_path: Exact directory containing the raw world metadata chunks.
        target: Raw target frame. Its shape must exactly match the stored
            frames in the raw world chunks.
        verbose: Whether to display chunk-level search progress.

    Returns:
        A tuple containing the matching metadata row's `AGCDgain` or legacy
        `Dgain` value and the zero-based global raw-frame index. Returns
        `None` when the target does not occur in the raw chunks.
    """

    target = np.asarray(target)
    if target.ndim not in (2, 3):
        raise ValueError("target must be a two-dimensional grayscale or three-dimensional RGB frame.")

    metadata = world_util.world_metadata_from_chunks(raw_chunks_path)
    metadata = metadata.dropna().reset_index(drop=True)
    dgain_column = next((name for name in ("AGCDgain", "Dgain") if name in metadata.columns), None)
    if dgain_column is None:
        raise KeyError("World metadata does not contain an 'AGCDgain' or legacy 'Dgain' column.")
    matching_frame_index = chunk_io.find_frame_index(
        raw_chunks_path,
        target,
        verbose=verbose,
    )
    if matching_frame_index is None:
        return None
    if matching_frame_index >= len(metadata):
        raise ValueError(
            f"Raw frame index {matching_frame_index} exceeds the {len(metadata)} non-NaN metadata rows."
        )

    dgain = float(metadata.iloc[matching_frame_index][dgain_column])
    return dgain, matching_frame_index

In [ ]:
example_frames_dir: str = "/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/data/exampleWorldCameraImages"

indoor_frames: np.ndarray = np.array([cv2.imread(os.path.join(example_frames_dir, frame), cv2.IMREAD_GRAYSCALE) for frame in natsorted(os.listdir(example_frames_dir)) if "indoor" in frame])
indoor_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"


for idx, frame in enumerate(indoor_frames, start=1):
    result = find_DGain(indoor_raw_path, frame)

    assert result is not None
    dgain, global_frame_idx = result
    print(f"Frame: {idx} has DGain: {dgain} at global frame index: {global_frame_idx}")

In [ ]:
example_frames_dir: str = "/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/data/exampleWorldCameraImages"

outdoor_frames: np.ndarray = np.array([cv2.imread(os.path.join(example_frames_dir, frame), cv2.IMREAD_GRAYSCALE) for frame in natsorted(os.listdir(example_frames_dir)) if "outdoor" in frame])
outdoor_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"


for idx, frame in enumerate(outdoor_frames, start=1):
    result = find_DGain(outdoor_raw_path, frame)

    assert result is not None
    dgain, global_frame_idx = result
    print(f"Frame: {idx} has DGain: {dgain} at global frame index: {global_frame_idx}")

## Usage

Load one of the raw TIFF files into a NumPy array, then pass it with its source raw-chunk path:

```python
target = ...  # np.ndarray loaded from indoor_1.tiff, outdoor_1.tiff, etc.
result = find_DGain(
    raw_chunks_path="/path/to/raw/chunks",
    target=target,
)
if result is not None:
    dgain, global_frame_idx = result
    print(dgain, global_frame_idx)
```